In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("=== Things in MyDrive ===")
for item in os.listdir('/content/drive/MyDrive'):
    print(f"  {item}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Things in MyDrive ===
  Colab Notebooks
  Assessment


In [ ]:
import os
print("=== Inside Assessment ===")
for item in os.listdir('/content/drive/MyDrive/Assessment'):
    print(f"  {item}")

=== Inside Assessment ===
  Sign Language Detection-20260416T180914Z-3-001
  7. Racist Sexist or Not Dataset-20260429T122954Z-3-001
  TextTask Models
  VisionTask Models


In [ ]:
import shutil, os

# Delete the partial copy
LOCAL_DIR = '/content/sign_language_local'
if os.path.exists(LOCAL_DIR):
    shutil.rmtree(LOCAL_DIR)
    print(' Deleted partial copy')

# Redo copy from scratch
SOURCE = '/content/drive/MyDrive/Assessment/Sign Language Detection-20260416T180914Z-3-001/Sign Language Detection'
print('Copying fresh (3-5 min)...')
shutil.copytree(SOURCE, LOCAL_DIR)

# Verify
TRAIN_DIR = f'{LOCAL_DIR}/Train'
class_count = len(os.listdir(TRAIN_DIR))
print(f'Train classes found: {class_count}')

if class_count == 38:
    print(' ALL 38 classes copied!')
else:
    print(f'Still only {class_count}. Try again.')

🗑️ Deleted partial copy
Copying fresh (3-5 min)...
Train classes found: 38
🎉 ALL 38 classes copied!


In [ ]:
# ============================================================
# COMPLETE PIPELINE — ABLATION + TRANSFER LEARNING
# ============================================================
import shutil, os, time, pickle
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# ── Setup ──────────────────────────────────────────────────
LOCAL_DIR = '/content/sign_language_local'
SOURCE = '/content/drive/MyDrive/Assessment/Sign Language Detection-20260416T180914Z-3-001/Sign Language Detection'

if not os.path.exists(LOCAL_DIR):
    print('Copying dataset (3-5 min)...')
    shutil.copytree(SOURCE, LOCAL_DIR)
    print('Done copying!')

TRAIN_DIR = f'{LOCAL_DIR}/Train'
print('Classes:', len(os.listdir(TRAIN_DIR)))

NUM_CLASSES, IMG_SIZE, BATCH_SIZE = 38, (224, 224), 32

train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
    rotation_range=15, zoom_range=0.15,
    width_shift_range=0.1, height_shift_range=0.1,
    brightness_range=[0.8, 1.2], fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=42)
val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=42, shuffle=False)

class PercentLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f'Epoch {epoch+1} | Train: {logs["accuracy"]*100:.2f}% | Val: {logs["val_accuracy"]*100:.2f}%')

SAVE_DIR = '/content/drive/MyDrive/Assessment/VisionTask Models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── ABLATION MODEL ─────────────────────────────────────────
print('\n========== TRAINING ABLATION (No Dropout) ==========')
def build_ablation(num_classes, input_shape=(224,224,3)):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32,(3,3),activation='relu',padding='same'), layers.BatchNormalization(),
        layers.Conv2D(32,(3,3),activation='relu',padding='same'), layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64,(3,3),activation='relu',padding='same'), layers.BatchNormalization(),
        layers.Conv2D(64,(3,3),activation='relu',padding='same'), layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128,(3,3),activation='relu',padding='same'), layers.BatchNormalization(),
        layers.Conv2D(128,(3,3),activation='relu',padding='same'), layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(256,(3,3),activation='relu',padding='same'), layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(512,activation='relu'), layers.BatchNormalization(),
        layers.Dense(256,activation='relu'),
        layers.Dense(num_classes,activation='softmax')
    ])

ablation_model = build_ablation(NUM_CLASSES)
ablation_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                        loss='categorical_crossentropy', metrics=['accuracy'])

t = time.time()
history_ablation = ablation_model.fit(
    train_generator, epochs=10, validation_data=val_generator,
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
               PercentLogger()], verbose=1)
print(f'Ablation time: {(time.time()-t)/60:.1f} min')
print(f'Ablation Train: {history_ablation.history["accuracy"][-1]*100:.2f}% | Val: {history_ablation.history["val_accuracy"][-1]*100:.2f}%')

# SAVE IMMEDIATELY
ablation_model.save(f'{SAVE_DIR}/ablation_no_dropout.keras')
with open(f'{SAVE_DIR}/ablation_history.pkl', 'wb') as f:
    pickle.dump(history_ablation.history, f)
print('✅ Ablation saved to Drive!')

# ── TRANSFER LEARNING ──────────────────────────────────────
print('\n========== TRAINING TRANSFER LEARNING ==========')
tl_train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input, validation_split=0.2,
    rotation_range=15, zoom_range=0.15,
    width_shift_range=0.1, height_shift_range=0.1,
    brightness_range=[0.8, 1.2], fill_mode='nearest')
tl_val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)

tl_train_gen = tl_train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=42)
tl_val_gen = tl_val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=42, shuffle=False)

# Phase 1: Feature Extraction
print('\n--- Phase 1: Feature Extraction ---')
base_model = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = keras.Input(shape=(224,224,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

tl_model = keras.Model(inputs, outputs)
tl_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='categorical_crossentropy', metrics=['accuracy'])

t = time.time()
history_tl1 = tl_model.fit(
    tl_train_gen, epochs=8, validation_data=tl_val_gen,
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
               PercentLogger()], verbose=1)
print(f'Phase 1 time: {(time.time()-t)/60:.1f} min')

# Phase 2: Fine-Tuning
print('\n--- Phase 2: Fine-Tuning ---')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

tl_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy', metrics=['accuracy'])

t = time.time()
history_tl2 = tl_model.fit(
    tl_train_gen, epochs=10, validation_data=tl_val_gen,
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
               PercentLogger()], verbose=1)
print(f'Phase 2 time: {(time.time()-t)/60:.1f} min')

# SAVE IMMEDIATELY
tl_model.save(f'{SAVE_DIR}/mobilenetv2_tl.keras')
with open(f'{SAVE_DIR}/tl_phase1_history.pkl', 'wb') as f:
    pickle.dump(history_tl1.history, f)
with open(f'{SAVE_DIR}/tl_phase2_history.pkl', 'wb') as f:
    pickle.dump(history_tl2.history, f)
print('✅ Transfer Learning saved to Drive!')

# ── FINAL SUMMARY ──────────────────────────────────────────
print('\n========== ALL DONE! ==========')
print(f'Ablation Final Val: {history_ablation.history["val_accuracy"][-1]*100:.2f}%')
print(f'TL Phase 1 Final Val: {history_tl1.history["val_accuracy"][-1]*100:.2f}%')
print(f'TL Phase 2 Final Val: {history_tl2.history["val_accuracy"][-1]*100:.2f}%')

Classes: 38
Found 8648 images belonging to 38 classes.
Found 2144 images belonging to 38 classes.

========== TRAINING ABLATION (No Dropout) ==========
Epoch 1/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 539ms/step - accuracy: 0.0362 - loss: 3.9761Epoch 1 | Train: 4.46% | Val: 2.94%
271/271 ━━━━━━━━━━━━━━━━━━━━ 179s 569ms/step - accuracy: 0.0446 - loss: 3.8504 - val_accuracy: 0.0294 - val_loss: 4.5102
Epoch 2/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - accuracy: 0.0869 - loss: 3.4934Epoch 2 | Train: 11.11% | Val: 6.11%
271/271 ━━━━━━━━━━━━━━━━━━━━ 138s 508ms/step - accuracy: 0.1111 - loss: 3.3299 - val_accuracy: 0.0611 - val_loss: 4.2482
Epoch 3/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step - accuracy: 0.2627 - loss: 2.4962Epoch 3 | Train: 34.01% | Val: 7.42%
271/271 ━━━━━━━━━━━━━━━━━━━━ 135s 499ms/step - accuracy: 0.3401 - loss: 2.2088 - val_accuracy: 0.0742 - val_loss: 5.4944
Epoch 4/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - accuracy: 0.5854 - loss: 1.3521Epoch 4 | Train: 62.23% | V